In [ ]:
%load_ext autoreload
%autoreload 2


from tqdm import tqdm
import glob
import os
from pathlib import Path
from labelbox import Client, MediaType
from datetime import datetime



from footy_track.object_detections import labelling
from footy_track.object_detections import detectors
from footy_track.object_detections.utils import visualise_detections_on_image
from footy_track.object_detections.schema import FrameDetections


DATA_DIR = Path("../data/arsenal_mancity_frames_1fps")

PROJECT_ID = None # Create a new project
PROJECT_ID="cmigodkqe01a607xd6dz3c7or"

DATASET_ID= None 
DATASET_ID="cmigoazqe00990739p78i3vrl"

In [ ]:
# Create ontology in Labelbox and print its ID
ontology_helper = labelling.SimplifiedOntology(id=labelling.SIMPLIFIED_ONTOLOGY_ID)

In [ ]:
all_image_paths = sorted(Path(DATA_DIR).glob("*.jpg"))

NUM_SAMPLES = 1000

# Sample 100 image paths across the image paths
image_path = all_image_paths
image_paths = all_image_paths[:: max(1, len(all_image_paths) // NUM_SAMPLES)]
len(image_paths)

In [ ]:
path_to_detections = {img_path: None  for img_path in image_paths}

In [ ]:
api_key = os.getenv("LABELBOX_API_KEY") or os.getenv("LB_API_KEY")
if not api_key:
    raise RuntimeError("Set LABELBOX_API_KEY (or LB_API_KEY) in your environment.")

client = Client(api_key=api_key)

In [ ]:
dataset_name = f"GroundingDINO - {Path(DATA_DIR).name} - {datetime.now():%Y-%m-%d %H:%M}"
if DATASET_ID :
    ds = client.get_dataset(DATASET_ID)
    print("Using existing dataset:", getattr(ds, "uid", None) or getattr(ds, "id", None), "-", ds.name)
else:
    ds = client.create_dataset(name=dataset_name)
    print("Created dataset:", getattr(ds, "uid", None) or getattr(ds, "id", None), "-", dataset_name)


# Build data rows and upload them. Use external_id/globalKey equal to filename so imports can reference them
rows_to_create = []
for img_path, det in path_to_detections.items():
    fname = img_path.stem
    row_data = str(img_path.resolve())
    rows_to_create.append({"row_data": row_data, "external_id": fname, "global_key": fname})

upload_job = ds.create_data_rows(rows_to_create)
print(f"Uploaded {len(rows_to_create)} data rows to dataset {dataset_name}.")

In [ ]:
if PROJECT_ID:
    project = client.get_project(PROJECT_ID)
else:
    project = client.create_project(name=f"{dataset_name}", media_type=MediaType.Image)


project.connect_ontology(ontology_helper.get_ontology())


In [ ]:

# Attach dataset via batch
rows = list(ds.data_rows())
project.create_batch(
    f"batch-{datetime.now():%Y-%m-%d %H:%M:%S}",
    rows,
    priority=1,
)

In [ ]:
# detector = detectors.GroundingDinoObjectDetector()
detector = detectors.UltralyticsObjectDetector(model_uri="yolo11x.pt")

detection = detector.predict_from_path(image_path=image_path[500])
visualise_detections_on_image(frame_detections=detection)

In [ ]:
def cache_path_from_img_path(img_path: Path) -> Path:
    return img_path.parent / "cached_detections" / (img_path.stem + ".json")
    
cache = True
for img_path, det in tqdm(path_to_detections.items(), total=len(path_to_detections)):
    cache_path = cache_path_from_img_path(img_path)
    if det is not None:
        pass
    elif cache and cache_path.exists():
        with open(cache_path) as f:
            det = FrameDetections.model_validate_json(cache_path.read_text())
    else:
        det = detector.predict_from_path(img_path)
    
    if cache:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_path, "w") as f:
            f.write(det.model_dump_json())
    
    path_to_detections[img_path] = det


In [ ]:
import labelbox.data.annotation_types as lb_types

def frame_to_object_annotation(det: FrameDetections, global_key: str) -> lb_types.Label:
    """Convert a FrameDetections into a Labelbox `Label` object."""
    annotations = []
    for d in det.detections:
        top = int(d.y * det.height)
        left = int(d.x * det.width)
        width_px = int(d.w * det.width)
        height_px = int(d.h * det.height)
        obj = lb_types.ObjectAnnotation(
            name=d.label, 
            value=lb_types.Rectangle(
                start=lb_types.Point(x=left, y=top),
                end=lb_types.Point(x=left + width_px, y=top + height_px)
            )
        )
        annotations.append(obj)
    data = {"global_key": global_key} if global_key else {}
    return lb_types.Label(data=data, annotations=annotations)

In [ ]:
labelbox_dataset = ds
print('Uploaded', len(rows_to_create), 'data rows')

# Now build label objects from FrameDetections and import them as pre-labels via LabelImport.create_from_objects
from labelbox.schema.annotation_import import LabelImport
labels = []
for img_path, det in path_to_detections.items():
    if not det or not det.detections:
        continue
    labels.append(frame_to_object_annotation(det,global_key=img_path.stem))

print('Prepared', len(labels), 'label objects for import')

In [ ]:
list(ds.data_rows())

In [ ]:
import uuid
from labelbox import MALPredictionImport

upload_job = MALPredictionImport.create_from_objects(
    client = client,
    project_id = project.uid,
    name = "mal_job"+str(uuid.uuid4()),
    predictions = labels,
)



In [ ]:
upload_job.wait_till_done()